# Physical AI 基础篇 智能机器人课程 2026 Spring 第2讲练习资料
## 观察 Kinematics（运动学）的内部工作方式

本 notebook 通过实际调用 Kinematics API，加深对运动学的理解。

具体来说，我们使用 Franka Panda 调用 FK / IK API，把自己定义的 task-space 轨迹交给 IK，再让机械臂运动起来。

关键数据流是：FK 把关节空间映射到末端位姿；IK 则把期望末端位姿翻译回一组可执行的关节角。


## 安装依赖

- `dm_control` …… IK 求解器
- `mujoco` …… 仿真器
- `robot_descriptions` …… 获取 Panda 的 MuJoCo 模型


In [ ]:
!apt-get update
!apt-get install -y \
    libegl1 libegl1-mesa-dev libgles2-mesa-dev \
    libglfw3 libosmesa6-dev libgl1-mesa-glx > /dev/null

!pip install "imageio[ffmpeg]" "dm_control==1.0.41" robot_descriptions "mujoco==3.8.1"

### Kinematics（运动学）
- **Joint space（关节空间）** `q = [θ₁, …]`：每个关节的角度
- **Task space（任务空间）** `p = [x, y, z, …]`：末端执行器的位姿
- **FK（正向运动学）** `p = f(q)`：关节角 → EE
- **IK（逆向运动学）** `q = f⁻¹(p)`：EE → 关节角，**通常没有解析解**


我们将在与真实机器人相当的 7 自由度 **Franka Emika Panda** 上实际调用 FK / IK。

本次在 **MuJoCo 中仿真 Panda**，并使用 **[dm_control](https://github.com/google-deepmind/dm_control)** 的 `qpos_from_site_pose` 求解逆运动学。


In [ ]:
import os, re
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import numpy as np
import mujoco
from dm_control import mujoco as dm_mujoco
from dm_control.utils import inverse_kinematics as ik
from robot_descriptions import panda_mj_description

# dm_control 的 IK 是「site 的姿态」目标。menagerie 的 Panda 中只有 site 没有的，、
# 末端(hand body)到 EE 用的 site 注意意入后重新加载
pkg_dir = os.path.dirname(panda_mj_description.MJCF_PATH)
xml = open(panda_mj_description.MJCF_PATH).read()
xml = re.sub(r'(<body\s+name="hand"[^>]*>)',
             r'\1\n      <site name="ee_site" pos="0 0 0" size="0.01" rgba="1 0 0 0"/>',
             xml, count=1)
open(os.path.join(pkg_dir, "_panda_ee.xml"), "w").write(xml)

scene_src = os.path.join(pkg_dir, "scene.xml")
if os.path.exists(scene_src):
    scene = open(scene_src).read().replace('"panda.xml"', '"_panda_ee.xml"')
    load_path = os.path.join(pkg_dir, "_scene_ee.xml"); open(load_path, "w").write(scene)
else:
    load_path = os.path.join(pkg_dir, "_panda_ee.xml")

physics = dm_mujoco.Physics.from_xml_path(load_path)
mj_model, mj_data = physics.model.ptr, physics.data.ptr

EE_SITE = "ee_site"
ARM_JOINTS = [f"joint{i}" for i in range(1, 8)]   # IK ，動还是的是机械臂 7 关节仅（手指固定）

# Panda 的标准 "home"(ready) 姿态。qpos 是 9 维度 = 机械臂 7 关节 + 指 2 关节。
q_full_home = np.array([0, 0, 0, -1.57079, 0, 1.57079, -0.7853, 0.04, 0.04])
q_home = q_full_home[:7].copy()

def full_q(q7):
    "机械臂 7 关节角 MuJoCo 的全部部 qpos（9 维度、指是 home 的）方向转换"
    q = q_full_home.copy(); q[:7] = np.asarray(q7, float)
    return q

def fk(q7):
    "Forward kinematics: 7 关节角 -> 末端 site 的位置(3) 与姿态四元几 wxyz(4)"
    physics.data.qpos[:] = full_q(q7)
    physics.forward()
    pos = physics.named.data.site_xpos[EE_SITE].copy()
    quat = np.empty(4)
    mujoco.mju_mat2Quat(quat, physics.named.data.site_xmat[EE_SITE])
    return pos, quat

start_pos, home_quat = fk(q_home)   # home 的末端位置与姿态

print("home 关节角 q (rad):", np.round(q_home, 3))
print("home 的末端位置 (m)   :", np.round(start_pos, 3))

准备绘图辅助函数（后面也会用于制作视频）。这次使用 **MuJoCo 的离屏渲染器**把 Panda 渲染成真实图像，收集帧并制作 mp4 视频。


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display
from PIL import Image

# MuJoCo 离屏・渲染器
renderer = mujoco.Renderer(mj_model, height=480, width=640)

# 見和斜前方从的相机
cam = mujoco.MjvCamera()
mujoco.mjv_defaultFreeCamera(mj_model, cam)
cam.lookat[:] = (0.2, 0.0, 0.4)
cam.distance = 1.8
cam.azimuth = 135
cam.elevation = -20

def render_scene(q_full, markers=None):
    "关节角 q_full（9 维度）的将姿态渲染成图像"
    mj_data.qpos[:] = q_full
    mujoco.mj_forward(mj_model, mj_data)
    renderer.update_scene(mj_data, cam)
    if markers is not None:
        scn = renderer.scene
        for p in markers:
            if scn.ngeom >= scn.maxgeom:
                break
            mujoco.mjv_initGeom(
                scn.geoms[scn.ngeom], int(mujoco.mjtGeom.mjGEOM_SPHERE),
                np.array([0.012, 0.0, 0.0]), np.asarray(p, float),
                np.eye(3).flatten(), np.array([0.9, 0.1, 0.1, 1.0], np.float32))
            scn.ngeom += 1
    return renderer.render()

def show_video(frames, name, fps=24, width=480):
    "将帧序列编码为 mp4 并显示"
    imageio.mimsave(name, frames, fps=fps)
    display(Video(name, embed=True, width=width))

def show_image(img):
    "一张 RGB 图像（或横向拼接的图像）并显示"
    display(Image.fromarray(np.asarray(img)))

### FK（正向运动学）：关节角 → EE

给 `fk(q)` 传入 7 个关节角，就会得到末端（EE）的**位置 `(x, y, z)`**。这就是 **joint space → task space** 的映射。无论输入多少种 `q`，每个 `q` 都会唯一决定一个末端位置。


In [ ]:
# Panda 的标准 "home"（ready）姿态，正向运动学确认
pos, _ = fk(q_home)

print("关节角 q (rad):", np.round(q_home, 3))
print("→ 末端位置 (m):", np.round(pos, 3))

# 该姿态 MuJoCo ，実画像与并渲染
show_image(render_scene(q_full_home))

试着只旋转**第一个关节 θ₁**，观察末端如何运动（改变一个关节后末端随之运动，这就是 FK）。


In [ ]:
# 第1关节 θ1 仅左右摆动、末端（与腕全部部体）如何运动制作视频
sweep = np.concatenate([
    np.linspace(0.0, 1.2, 9),
    np.linspace(1.2, -1.2, 18),
    np.linspace(-1.2, 0.0, 9),
])
frames = []
pos_list = []
for dq in sweep:
    q7 = q_home.copy()
    q7[0] += dq          # θ1 仅
    frames.append(render_scene(full_q(q7)))
    pos, _ = fk(q7)
    pos_list.append(pos)

show_video(frames, "fk_joint1.mp4")
pos_list = np.asarray(pos_list)
pos_list

### IK（逆向运动学）：EE → 关节角

`ik_reach(p)` 接收 **EE 的目标位置**，返回实现该位置的**关节角**。内部调用的是 `dm_control.utils.inverse_kinematics.qpos_from_site_pose`。

验证 **`FK(IK(p*)) ≈ p*`**：把 IK 反算出的关节角重新送入正向运动学，结果应当接近目标位置。


In [ ]:
def ik_reach(target_pos, target_quat=None, seed=q_home):
    physics.data.qpos[:] = full_q(seed)
    physics.forward()      # 探索的初期姿态 seed 到
    res = ik.qpos_from_site_pose(
        physics,
        EE_SITE,
        target_pos=np.asarray(target_pos, float),
        target_quat=home_quat if target_quat is None else target_quat,
        joint_names=ARM_JOINTS,                                 # 机械臂 7 关节仅動还是
        tol=1e-4, max_steps=200
    )
    return res.qpos, res.success

# home 的末端从稍微完成位置目标
target_pos = start_pos + np.array([0.0, 0.15, -0.10])
q_sol, ok = ik_reach(target_pos)
achieved, _ = fk(q_sol[:7])
err = float(np.linalg.norm(achieved - target_pos))

print("目标的末端位置 p*    :", np.round(target_pos, 3))
print("IK 返完成关节角 q* :", np.round(q_sol[:7], 3))
print("FK(q*) ，恢复       :", np.round(achieved, 3))
print(f"残差 ‖FK(IK(p*)) − p*‖ = {err*1000:.3f} mm  (success={ok})")
show_image(render_scene(q_sol))

### 让 EE 移动到目标位置

人先设计“末端要怎样运动”的 task-space 轨迹，再用 **dm_control 的 IK 将每个 waypoint 翻译成关节角**并跟踪，最后通过视频确认 Panda 的运动。

设计的运动：**向前移动（+x 方向 0.1 m）→ 向下移动（−z 方向 0.1 m）**


In [ ]:
# 轨迹设计（task space）: 向前(+x 0.1m) → 向下(−z 0.1m)
seg1 = [start_pos + np.array([0.10 * t, 0, 0]) for t in np.linspace(0, 1, 12)]   # +x 方向
seg2 = [seg1[-1] + np.array([0, 0, -0.10 * t]) for t in np.linspace(0, 1, 12)]   # −z 方向
ee_path = seg1 + seg2

def track_path(positions, orientation=None, markers=None):
    seed = q_home                                  # 最初的通過点是 home 从开始
    frames, errors = [], []
    for p in positions:
        p = np.asarray(p, float)
        q_full, _ = ik_reach(p, target_quat=orientation, seed=seed)
        seed = q_full[:7]                          # 下一次通過点是今的解从开始
        achieved, _ = fk(q_full[:7])
        errors.append(np.linalg.norm(achieved - p))
        frames.append(render_scene(q_full, markers=markers))
    return frames, np.asarray(errors)

frames, errors = track_path(ee_path, markers=ee_path)
print(f"設計完成末端轨迹的waypoint 几: {len(ee_path)}")
print(f"末端的跟踪误差: 平均 {errors.mean()*1000:.2f} mm / 最大 {errors.max()*1000:.2f} mm")
show_video(frames, "handwritten_traj.mp4", fps=12)

我们只在 **task space 中指定“这样运动”**，然后由 **IK（`dm_control` 的 `qpos_from_site_pose`）把它翻译成关节角**，Panda 就在 MuJoCo 中按设计运动起来了，跟踪误差也保持很小。

这说明 task-space 规划与 joint-space 控制之间的桥梁，就是 IK 求解器。
